In [ ]:
import pandas as pd
import numpy as np
import glob
import os

CSV_FOLDER = r"../CSV Dataset"

csv_files = glob.glob(os.path.join(CSV_FOLDER, '*.csv'))

if not csv_files:
    print(f"No CSV files found in: {CSV_FOLDER}")
    exit()

for file in csv_files:
    print(f"Processing {file}...")
    df = pd.read_csv(file)

    x_cols = [c for c in df.columns if c.startswith('x')]
    y_cols = [c for c in df.columns if c.startswith('y')]
    z_cols = [c for c in df.columns if c.startswith('z')]

    mask = (df[x_cols + y_cols + z_cols] != -1).all(axis=1)
    df = df[mask]

    # normalize relative to face center
    face_center_x = df[x_cols].mean(axis=1)
    face_center_y = df[y_cols].mean(axis=1)

    for col in x_cols:
        df[col] = df[col] - face_center_x
    for col in y_cols:
        df[col] = df[col] - face_center_y

    # scale per-frame so values fit in [-1, 1]
    max_per_frame = pd.concat([df[x_cols].abs(), df[y_cols].abs()], axis=1).max(axis=1)
    max_per_frame = max_per_frame.replace(0, 1)  # Guard against division by zero

    for col in x_cols:
        df[col] = df[col] / max_per_frame
    for col in y_cols:
        df[col] = df[col] / max_per_frame

    # drop z-coordinates
    df = df.drop(columns=z_cols)

    # save
    base = os.path.basename(file)
    name, ext = os.path.splitext(base)
    out_file = os.path.join(CSV_FOLDER, f"{name}_cleaned{ext}")

    df.to_csv(out_file, index=False)
    print(f"Saved cleaned file to {out_file}")

print("\nDone.")